In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, precision_recall_fscore_support

In [ ]:
low_priority_tickets = [
    "Need help updating my profile picture on the portal",
    "How do I reset my account password regularly",
    "Where can I find the monthly billing invoice",
    "General inquiry regarding the software feature roadmap",
    "Typo observed in the footer documentation section",
    "Requesting read access to the shared folder archive",
    "Change my registered email address in settings",
    "Question about how to export user list to csv"
] * 40  # 320 records

urgent_priority_tickets = [
    "Critical server down production outage blocking users",
    "Fatal database crash system unrecoverable error urgently",
    "Data breach suspected unauthorized access detected immediately",
    "Payment gateway down all customer transactions failing"
] * 20  # 80 records



In [ ]:
# -------------------------------------------------------------
# 1. Generate Imbalanced Dataset (e.g., Support Ticket Priority)
# -------------------------------------------------------------
# Low priority: 80% (Class 0), Urgent priority: 20% (Class 1)



texts = low_priority_tickets + urgent_priority_tickets
labels = [0] * len(low_priority_tickets) + [1] * len(urgent_priority_tickets)

df = pd.DataFrame({"ticket_text": texts, "is_urgent": labels})


In [ ]:



# -------------------------------------------------------------
# 3. Feature Extraction (TF-IDF with Non-Negative Values)
# -------------------------------------------------------------
# Complement NB requires non-negative counts or TF-IDF values
vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True)
X = vectorizer.fit_transform(df["ticket_text"])
y = df["is_urgent"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# -------------------------------------------------------------
# 4. Model Training & Evaluation
# -------------------------------------------------------------
# norm=True applies Second-step weight normalization
model = ComplementNB(alpha=1.0, norm=True)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("=== Complement NB Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Standard", "Urgent"]))

# -------------------------------------------------------------
# 5. Model Diagnostics Visualizations
# -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Visual 1: Normalized Confusion Matrix (Standard vs. Complement minority performance)
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred, 
    display_labels=["Standard", "Urgent"], 
    normalize="true", 
    cmap="YlOrRd", 
    ax=axes[0]
)
axes[0].set_title("Normalized Confusion Matrix\n(Shows Recall per Class)")

# Visual 2: Metric Performance Across Classes
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred)
metrics_df = pd.DataFrame({
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1
}, index=["Standard", "Urgent"])

metrics_df.plot(kind="bar", ax=axes[1], colormap="viridis")
axes[1].set_title("Model Metrics Across Imbalanced Classes")
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("Score")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()